# 🌿 Notebook 1: Plant Biopotential Time-Series EDA & Feature Engineering

## 1. Project Background & Electrophysiology
Plant electrophysiology measures electrical potential differences across plant tissues generated by ion transport (principally $Ca^{2+}$, $K^+$, and $Cl^-$ fluxes) across cell membranes.
Using an **AD8232 analog front-end biopotential amplifier** connected to an Arduino and electrodes attached to plant leaves and stems, electrical potential readings (ADC counts, 0–1023) are recorded at high temporal resolution.

### Scientific Objectives:
1. **Signal Inspection:** Characterize the raw plant biopotential waveforms under different stimuli (`normal`, `music`, `water`, `dead`).
2. **Frequency Spectrum Analysis:** Compute the Power Spectral Density (PSD) using Fast Fourier Transforms (FFT) and Welch periodograms.
3. **Temporal Dynamics:** Evaluate rolling variance, signal volatility, and moving baseline drift.
4. **Time-Series Feature Extraction:** Segment continuous recordings into temporal sliding windows ($W = 200$ samples) and compute statistical, morphological, and spectral features.
5. **Standardized Dataset Export:** Save `plant_time_series_features.csv` to serve as the unified benchmark dataset for all subsequent machine learning notebooks.


In [1]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats, signal

# Set presentation-grade styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print("Libraries imported successfully!")


ModuleNotFoundError: No module named 'pandas'

## 2. Load & Explore Time-Series Sessions Across Categories
We scan [`../categorize csv`](../categorize%20csv) to discover all valid recording files.


In [ ]:
DATA_ROOT = os.path.join("..", "categorize csv")
if not os.path.exists(DATA_ROOT):
    DATA_ROOT = "categorize csv"

categories = ["normal", "music/classical", "music/devotional", "music/pop", "music/general", "water", "dead"]
file_inventory = []

for cat in categories:
    cat_dir = os.path.join(DATA_ROOT, os.path.normpath(cat))
    csv_files = glob.glob(os.path.join(cat_dir, "*.csv"))
    for f in csv_files:
        try:
            row_count = sum(1 for _ in open(f, encoding='utf-8', errors='ignore')) - 1
            if row_count > 50:
                file_inventory.append({
                    "filepath": f,
                    "filename": os.path.basename(f),
                    "category": cat,
                    "rows": row_count
                })
        except Exception:
            pass

inventory_df = pd.DataFrame(file_inventory)
print(f"Total valid time-series sessions: {len(inventory_df)}")
print(inventory_df.groupby("category")["rows"].agg(["count", "sum", "mean"]))


## 3. Raw Waveform Comparison Across Stimuli
Let's visualize sample time-series waveforms across four representative states:
1. **Normal / Baseline:** Stable resting potential with micro-fluctuations.
2. **Watering Event:** Ion transport triggering depolarization shifts.
3. **Music Stimulation:** Acoustic pressure waves modulating biopotential dynamics.
4. **Dead Plant Tissue:** Lack of membrane polarization, exhibiting flatline thermal noise.


In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=False)
sample_map = {
    "Normal Baseline": ("normal", "#1976D2"),
    "Devotional Music": ("music/devotional", "#7B1FA2"),
    "Water Exposure": ("water", "#388E3C"),
    "Dead Plant Tissue": ("dead", "#D32F2F")
}

for ax, (title, (cat, color)) in zip(axes, sample_map.items()):
    sample_file = inventory_df[inventory_df["category"] == cat]["filepath"].iloc[0]
    df_sample = pd.read_csv(sample_file, nrows=1500)
    signal_vals = df_sample["ecg"].values
    time_pts = np.arange(len(signal_vals))
    
    # Plot raw signal
    ax.plot(time_pts, signal_vals, color=color, alpha=0.8, linewidth=1.2, label="Raw Biopotential (ADC)")
    
    # Overlay rolling moving average (smoothing window = 30)
    rolling_mean = pd.Series(signal_vals).rolling(window=30, center=True).mean()
    ax.plot(time_pts, rolling_mean, color="black", linestyle="--", linewidth=1.5, label="Filtered Baseline")
    
    ax.set_title(f"{title} - File: {os.path.basename(sample_file)[:40]}...", fontweight='bold')
    ax.set_ylabel("ADC Count (0-1023)")
    ax.set_ylim(-20, 1050)
    ax.legend(loc="upper right", framealpha=0.9)

axes[-1].set_xlabel("Time Samples (Observations)")
plt.tight_layout()
plt.show()


## 4. Spectral Domain: Power Spectral Density (PSD) & FFT Analysis
Plants respond to slow bio-electrical shifts as well as higher frequency environmental coupling.
We perform Fast Fourier Transform (FFT) and Welch's PSD estimation to uncover the dominant frequency bands.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

for title, (cat, color) in sample_map.items():
    sample_file = inventory_df[inventory_df["category"] == cat]["filepath"].iloc[0]
    df_sample = pd.read_csv(sample_file, nrows=2500)
    sig = df_sample["ecg"].values - np.mean(df_sample["ecg"].values)
    
    # Welch PSD
    freqs, psd = signal.welch(sig, nperseg=min(512, len(sig)))
    ax1.semilogy(freqs, psd, label=title, color=color, linewidth=1.8)
    
    # FFT Amplitude Spectrum
    fft_vals = np.abs(np.fft.rfft(sig))
    fft_freqs = np.fft.rfftfreq(len(sig))
    ax2.plot(fft_freqs[:200], fft_vals[:200], label=title, color=color, linewidth=1.5, alpha=0.85)

ax1.set_title("Welch Power Spectral Density (PSD)", fontweight='bold')
ax1.set_xlabel("Normalized Frequency")
ax1.set_ylabel("Power Density [ADC^2/Hz]")
ax1.legend()
ax1.grid(True, which='both', linestyle=':')

ax2.set_title("Low-Frequency FFT Amplitude Spectrum", fontweight='bold')
ax2.set_xlabel("Normalized Frequency")
ax2.set_ylabel("Magnitude")
ax2.legend()
ax2.grid(True, linestyle=':')

plt.tight_layout()
plt.show()


## 5. Rolling Volatility & Dynamic Variance Analysis
A live plant responding to external stimuli exhibits elevated signal variance and burst activity. Dead tissue exhibits low thermal/quantization noise.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for title, (cat, color) in sample_map.items():
    sample_file = inventory_df[inventory_df["category"] == cat]["filepath"].iloc[0]
    df_sample = pd.read_csv(sample_file, nrows=2000)
    rolling_std = pd.Series(df_sample["ecg"]).rolling(window=100).std()
    ax.plot(rolling_std.values, label=f"{title} (Rolling Std)", color=color, linewidth=1.6)

ax.set_title("Biopotential Volatility Over Time (Rolling Window = 100 samples)", fontweight='bold')
ax.set_xlabel("Sample Index")
ax.set_ylabel("Rolling Standard Deviation (ADC)")
ax.legend(loc='upper right')
ax.grid(True)
plt.tight_layout()
plt.show()


## 6. Time-Series Sliding Window Feature Engineering Pipeline

Because biopotential recordings are continuous time series, feeding raw point-by-point values directly into machine learning models ignores temporal context and leads to overfitting.
Instead, we segment each recording into **tumbling windows of $W = 200$ samples** and extract a comprehensive **21-dimensional feature vector**:

| Domain | Extracted Features | Biological & Mathematical Rationale |
| :--- | :--- | :--- |
| **Statistical Levels** | `mean`, `median`, `min`, `max`, `range`, `p25`, `p75`, `iqr` | Baseline biopotential level & steady-state polarization |
| **Dispersion & Shape** | `std`, `var`, `rms`, `skewness`, `kurtosis` | Signal amplitude spread, asymmetry, and impulsive spike likelihood |
| **Dynamics & Velocity** | `mad` (mean absolute difference), `zcr` (zero-crossing rate), `peak_count` | Rate of depolarization changes, frequency of threshold crossings |
| **Spectral Power** | `low_freq_power`, `mid_freq_power`, `high_freq_power`, `dominant_freq`, `spectral_entropy` | Frequency band distribution: slow vasomotor shifts vs fast vibration |


In [ ]:
def extract_window_features(sig):
    N = len(sig)
    if N < 10:
        return None
    
    mean_val = float(np.mean(sig))
    std_val = float(np.std(sig))
    var_val = float(np.var(sig))
    min_val = float(np.min(sig))
    max_val = float(np.max(sig))
    range_val = max_val - min_val
    median_val = float(np.median(sig))
    rms_val = float(np.sqrt(np.mean(sig**2)))
    mad_val = float(np.mean(np.abs(np.diff(sig)))) if N > 1 else 0.0
    
    p25, p75 = np.percentile(sig, [25, 75])
    iqr_val = float(p75 - p25)
    
    zero_crossings = int(np.sum(np.diff(sig > mean_val) != 0))
    zcr_val = zero_crossings / (N - 1) if N > 1 else 0.0
    
    skew_val = float(stats.skew(sig)) if std_val > 1e-5 else 0.0
    kurt_val = float(stats.kurtosis(sig)) if std_val > 1e-5 else 0.0
    
    threshold = mean_val + std_val
    peaks, _ = signal.find_peaks(sig, height=threshold)
    peak_count = len(peaks)
    
    centered = sig - mean_val
    fft_vals = np.abs(np.fft.rfft(centered))**2
    total_power = np.sum(fft_vals) + 1e-9
    
    num_bins = len(fft_vals)
    b1 = int(num_bins * 0.25)
    b2 = int(num_bins * 0.50)
    
    low_power = float(np.sum(fft_vals[:b1]) / total_power)
    mid_power = float(np.sum(fft_vals[b1:b2]) / total_power)
    high_power = float(np.sum(fft_vals[b2:]) / total_power)
    
    norm_fft = fft_vals / total_power
    norm_fft = norm_fft[norm_fft > 0]
    spectral_entropy = float(-np.sum(norm_fft * np.log2(norm_fft + 1e-12)))
    dominant_freq = int(np.argmax(fft_vals)) if len(fft_vals) > 0 else 0
    
    return {
        "mean": mean_val, "std": std_val, "var": var_val,
        "min": min_val, "max": max_val, "range": range_val,
        "median": median_val, "rms": rms_val, "mad": mad_val,
        "p25": float(p25), "p75": float(p75), "iqr": iqr_val,
        "zcr": zcr_val, "skew": skew_val, "kurt": kurt_val,
        "peak_count": peak_count,
        "low_freq_power": low_power, "mid_freq_power": mid_power, "high_freq_power": high_power,
        "spectral_entropy": spectral_entropy, "dominant_freq": dominant_freq
    }

print("Feature extractor defined with 21 engineered time-series features.")


## 7. Execute Feature Extraction Across All Sessions & Export Benchmark Dataset
We iterate through all valid files, windowing at $W = 200$, and building the master dataset `plant_time_series_features.csv`.


In [ ]:
WINDOW_SIZE = 200
features_records = []

for idx, row in inventory_df.iterrows():
    fpath = row["filepath"]
    category = row["category"]
    filename = row["filename"]
    
    if "dead" in category:
        main_cat = "dead"
        is_alive = 0
    elif "water" in category:
        main_cat = "water"
        is_alive = 1
    elif "music" in category:
        main_cat = "music"
        is_alive = 1
    else:
        main_cat = "normal"
        is_alive = 1
        
    df_file = pd.read_csv(fpath)
    if "ecg" not in df_file.columns or len(df_file) < WINDOW_SIZE:
        continue
        
    ecg_data = df_file["ecg"].values
    time_of_day = df_file["time_of_day"].iloc[0] if "time_of_day" in df_file.columns else "morning"
    
    num_windows = len(ecg_data) // WINDOW_SIZE
    max_windows = min(num_windows, 300)
    
    for w in range(max_windows):
        window_sig = ecg_data[w*WINDOW_SIZE : (w+1)*WINDOW_SIZE]
        feats = extract_window_features(window_sig)
        if feats is not None:
            feats["session_id"] = filename
            feats["category"] = category
            feats["main_category"] = main_cat
            feats["time_of_day"] = time_of_day
            feats["is_alive"] = is_alive
            feats["window_idx"] = w
            features_records.append(feats)

features_df = pd.DataFrame(features_records)
print(f"Extracted {len(features_df)} time-series feature windows across {features_df['session_id'].nunique()} sessions.")
print("\nDistribution across main categories:")
print(features_df["main_category"].value_counts())

# Save master dataset for all subsequent notebooks
features_df.to_csv("plant_time_series_features.csv", index=False)
print("\nSaved master feature dataset -> 'plant_time_series_features.csv'")


## 8. Feature Correlation & Pairwise Analysis
Let's inspect the correlation matrix among the engineered features to evaluate multicollinearity and feature diversity.


In [ ]:
feature_cols = [
    "mean", "std", "range", "rms", "mad", "zcr", 
    "skew", "kurt", "peak_count", "low_freq_power", 
    "high_freq_power", "spectral_entropy"
]

plt.figure(figsize=(11, 8))
corr = features_df[feature_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, square=True)
plt.title("Correlation Matrix of Engineered Time-Series Features", fontweight='bold')
plt.tight_layout()
plt.show()


## 9. Distribution of Key Features Across Categories
Violin plots illustrating how signal spread (`std`), signal velocity (`mad`), and frequency entropy separate living and stimulated states from dead tissue.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = [("std", "Signal Standard Deviation (Spread)"), 
           ("mad", "Mean Absolute Difference (Signal Velocity)"), 
           ("spectral_entropy", "Spectral Entropy (Frequency Complexity)")]

for ax, (m_col, m_title) in zip(axes, metrics):
    sns.violinplot(data=features_df, x="main_category", y=m_col, ax=ax, palette="Set2")
    ax.set_title(m_title, fontweight='bold')
    ax.set_xlabel("Stimulus Class")
    ax.grid(True, linestyle=':')

plt.tight_layout()
plt.show()


## 10. Summary & Handoff to Modeling Notebooks
✅ **Time-series signal inspection complete:** Raw biopotential dynamics and frequency signatures visualized.  
✅ **Feature Extraction complete:** 21 time- and frequency-domain metrics captured over sliding windows.  
✅ **Master dataset exported:** `plant_time_series_features.csv` is now ready to power Notebooks 2 through 6!
